# Exploratory Data Analysis


In [29]:
# Install required packages
%pip install nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nltk]2/3 [nltk]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /Users/yash/tf-venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install xgboost


  Using cached xgboost-3.1.2-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-3.1.2-py3-none-macosx_12_0_arm64.whl (2.2 MB)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /Users/yash/tf-venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:

import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

from sklearn.metrics.pairwise import cosine_similarity




In [2]:
# Load the Reviews dataset
reviews_df = pd.read_csv('Reviews.csv')


In [3]:
# Display basic information about the dataset
print("Dataset Shape:", reviews_df.shape)
print("\nColumn Names:")
print(reviews_df.columns.tolist())
print("\nFirst few rows:")
reviews_df.head()


Dataset Shape: (30000, 15)

Column Names:
['id', 'brand', 'categories', 'manufacturer', 'name', 'reviews_date', 'reviews_didPurchase', 'reviews_doRecommend', 'reviews_rating', 'reviews_text', 'reviews_title', 'reviews_userCity', 'reviews_userProvince', 'reviews_username', 'user_sentiment']

First few rows:


,id,brand,categories,manufacturer,name,reviews_date,reviews_didPurchase,reviews_doRecommend,reviews_rating,reviews_text,reviews_title,reviews_userCity,reviews_userProvince,reviews_username,user_sentiment
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",Universal Music Group / Cash Money,Pink Friday: Roman Reloaded Re-Up (w/dvd),2012-11-30T06:21:45.000Z,NaN,NaN,5,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,joshua,Positive
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor. This review was collected as part...,Good,NaN,NaN,dorothy w,Positive
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor.,Good,NaN,NaN,dorothy w,Positive
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-01-06T00:00:00.000Z,False,False,1,I read through the reviews on here before look...,Disappointed,NaN,NaN,rebecca,Negative
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-12-21T00:00:00.000Z,False,False,1,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,walker557,Negative


In [4]:
# Display dataset info and summary statistics
reviews_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    30000 non-null  object
 1   brand                 30000 non-null  object
 2   categories            30000 non-null  object
 3   manufacturer          29859 non-null  object
 4   name                  30000 non-null  object
 5   reviews_date          29954 non-null  object
 6   reviews_didPurchase   15932 non-null  object
 7   reviews_doRecommend   27430 non-null  object
 8   reviews_rating        30000 non-null  int64 
 9   reviews_text          30000 non-null  object
 10  reviews_title         29810 non-null  object
 11  reviews_userCity      1929 non-null   object
 12  reviews_userProvince  170 non-null    object
 13  reviews_username      29937 non-null  object
 14  user_sentiment        29999 non-null  object
dtypes: int64(1), object(14)
memory usage

In [5]:
# Display summary statistics for numerical columns
reviews_df.describe()


,reviews_rating
count,30000.000000
mean,4.483133
std,0.988441
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


# Data Cleaning 

#### Removing columns which are not necessary for our case study 

In [6]:
drop_cols = [
    "reviews_userProvince",
    "reviews_userCity",
    "reviews_didPurchase",
    "reviews_date",
    "manufacturer",
    "reviews_doRecommend"
]

reviews_df = reviews_df.drop(columns=drop_cols)

In [7]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                30000 non-null  object
 1   brand             30000 non-null  object
 2   categories        30000 non-null  object
 3   name              30000 non-null  object
 4   reviews_rating    30000 non-null  int64 
 5   reviews_text      30000 non-null  object
 6   reviews_title     29810 non-null  object
 7   reviews_username  29937 non-null  object
 8   user_sentiment    29999 non-null  object
dtypes: int64(1), object(8)
memory usage: 2.1+ MB


#### Handling the missing values in columns

Only three columns are having missing values
- reviews_title
- reviews_username
- user_sentiment


We need to drop rows which are having missing values for all the columns

In [8]:
reviews_df = reviews_df.dropna(
    subset=['reviews_title', 'reviews_username', 'user_sentiment']
)

reviews_df = reviews_df.reset_index(drop=True)

In [9]:
reviews_df.info()
reviews_df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29747 entries, 0 to 29746
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                29747 non-null  object
 1   brand             29747 non-null  object
 2   categories        29747 non-null  object
 3   name              29747 non-null  object
 4   reviews_rating    29747 non-null  int64 
 5   reviews_text      29747 non-null  object
 6   reviews_title     29747 non-null  object
 7   reviews_username  29747 non-null  object
 8   user_sentiment    29747 non-null  object
dtypes: int64(1), object(8)
memory usage: 2.0+ MB


id                  0
brand               0
categories          0
name                0
reviews_rating      0
reviews_text        0
reviews_title       0
reviews_username    0
user_sentiment      0
dtype: int64

# Exploratory Data Analysis

#### Sentiment Distribution

In [10]:
reviews_df['user_sentiment'].value_counts(normalize=True)


user_sentiment
Positive    0.887888
Negative    0.112112
Name: proportion, dtype: float64

Key Findings

- Dataset is highly imbalanced
- Accuracy is NOT a reliable metric
- We must focus on Precision, Recall, F1 Score and Confusion Matrix


Actions to be taken while modelling

- Use class imbalance handling during modeling
- class_weight='balanced' (Logistic Regression)
- scale_pos_weight (XGBoost)
- Stratified train-test split

#### Ratings Distribution

In [11]:
reviews_df['reviews_rating'].value_counts().sort_index()


reviews_rating
1     1361
2      409
3     1332
4     5992
5    20653
Name: count, dtype: int64

Key Findings

- Users overwhelmingly give 4–5 star ratings
- Typical e-commerce bias (happy users review more)
- Ratings alone are not enough to infer sentiment
- Text-based sentiment analysis is justified and necessary

#### Rating vs Sentiment consistency

In [12]:
pd.crosstab(reviews_df['reviews_rating'], reviews_df['user_sentiment'])


user_sentiment,Negative,Positive
reviews_rating,,
1,585,776
2,137,272
3,217,1115
4,548,5444
5,1848,18805


Key Findings

- 5-star negative reviews exist (1,848 rows!)
- 1-star positive reviews exist (776 rows!)
- This justifies using text sentiment, not rating-based heuristics

##### What this EDA tells us about MODELING

Sentiment Model

- Binary classification
- Imbalanced dataset
- Text-driven signal

Best Suited Models

- Logistic Regression (TF-IDF + class_weight)
- Naive Bayes (fast baseline)
- XGBoost (with careful tuning)

# Text Preprocessing

Combine review title + review text

In [13]:
reviews_df['review_combined'] = (
    reviews_df['reviews_title'] + " " + reviews_df['reviews_text']
)

Define a clean text preprocessing function

We will:
- Lowercase
- Remove punctuation & digits
- Remove stopwords
- Lemmatize words

In [14]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /Users/yash/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/yash/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [16]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove punctuation and digits
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize and remove stopwords + lemmatize
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    
    return " ".join(tokens)

In [17]:
reviews_df['clean_review'] = reviews_df['review_combined'].apply(clean_text)

In [18]:
reviews_df['sentiment_label'] = reviews_df['user_sentiment'].map({
    'Positive': 1,
    'Negative': 0
})


# Feature Extraction

Using TF-IDF for Feature Extraction because

- Works extremely well for sentiment analysis
- Reduces impact of common words
- Efficient & interpretable
- Perfect for Logistic Regression & Naive Bayes

In [19]:
X = reviews_df['clean_review']
y = reviews_df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

TF-IDF Vectorization

In [20]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

These parameters are very important:

- max_features=5000 → avoids overfitting
- ngram_range=(1,2) → captures phrases like “not good”
- min_df=5 → removes rare noisy words

At this point we have

- Cleaned text
- Numerical feature matrix
- Balanced train–test split
- Ready-to-train ML models

# Model Training

#### Train Logistic Regression Model

In [21]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
y_pred = lr_model.predict(X_test_tfidf)

In [23]:
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

              precision    recall  f1-score   support

    Negative       0.43      0.83      0.56       667
    Positive       0.98      0.86      0.91      5283

    accuracy                           0.86      5950
   macro avg       0.70      0.84      0.74      5950
weighted avg       0.91      0.86      0.88      5950



In [24]:
# Confusion Matrix

confusion_matrix(y_test, y_pred)

array([[ 552,  115],
       [ 736, 4547]])

##### Logistic Regression — Result Interpretation

Classification Report (Key Takeaways)

Negative class (minority, most important)

- Recall: 0.83  → 83% of negative reviews are correctly detected
- Precision: 0.43  → Many false positives
- F1-score: 0.56 → Decent given 11% class share

This is actually good performance for an imbalanced NLP problem.

Positive class (majority)

- Precision: 0.98
- Recall: 0.86
- F1-score: 0.91

Model is very strong on positive sentiment.

- Accuracy: 86%
- Macro F1: 0.74 → Important (treats classes equally)
- Weighted F1: 0.88

Confusion Matrix (Key Takeaways)

- Model catches most negatives (very important)
- Some positives are misclassified as negative

This trade-off is acceptable and often desirable in sentiment-based filtering
It’s better to mistakenly flag a positive as negative
than to miss a truly negative review.

#### Train Naive Bayes Model

In [25]:
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [26]:
y_pred_nb = nb_model.predict(X_test_tfidf)


In [27]:
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=['Negative', 'Positive']
))


              precision    recall  f1-score   support

    Negative       0.48      0.12      0.19       667
    Positive       0.90      0.98      0.94      5283

    accuracy                           0.89      5950
   macro avg       0.69      0.55      0.57      5950
weighted avg       0.85      0.89      0.86      5950



In [28]:
confusion_matrix(y_test, y_pred_nb)

array([[  81,  586],
       [  87, 5196]])

Naive Bayes — Result Interpretation

Classification Report (Key Insights)

Negative class (minority, CRITICAL)

- Recall: 0.12 
→ Only 12% of negative reviews detected
- Precision: 0.48
- F1-score: 0.19 

This is a deal-breaker for our use case.

Positive class

- Recall: 0.98 ✅
- Precision: 0.90 ✅
- F1-score: 0.94 ✅

As expected, Naive Bayes heavily favors the majority class.

Confusion Matrix Breakdown

586 out of 667 negative reviews were misclassified as positive

The model is essentially saying:

“Almost everything is positive”

This defeats the purpose of sentiment-based filtering.

Conclusion - Although Naive Bayes achieved higher overall accuracy, it failed to capture the minority negative sentiment class. Logistic Regression, with class weighting, provided significantly better recall for negative reviews, making it more suitable for sentiment-based recommendation filtering.

#### Train XGBoost Model

In [29]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()

scale_pos_weight = neg_count / pos_count
scale_pos_weight

0.1262719485067916

In [30]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_tfidf, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [31]:
y_pred_xgb = xgb_model.predict(X_test_tfidf)


In [32]:
print(classification_report(
    y_test,
    y_pred_xgb,
    target_names=['Negative', 'Positive']
))


              precision    recall  f1-score   support

    Negative       0.39      0.84      0.54       667
    Positive       0.98      0.84      0.90      5283

    accuracy                           0.84      5950
   macro avg       0.69      0.84      0.72      5950
weighted avg       0.91      0.84      0.86      5950



In [33]:
confusion_matrix(y_test, y_pred_xgb)


array([[ 560,  107],
       [ 858, 4425]])

Classification Report (Key Points)

Negative class (minority, most important)

- Recall: 0.84 (very high)
- Precision: 0.39 
- F1-score: 0.54

XGBoost is very similar to Logistic Regression in behavior:

Aggressively captures negatives at the cost of more false positives

Positive class

- Precision: 0.98
- Recall: 0.84
- F1-score: 0.90

Slightly worse than Logistic Regression on positives.

Confusion Matrix Analysis

Interpretation

Captures most negative reviews
Misclassifies more positives as negative compared to Logistic Regression

##### Among Logistic Regression, Naive Bayes, and XGBoost, Logistic Regression achieved the best balance between recall and precision for the minority negative sentiment class. Since the objective was to prevent negatively perceived products from being recommended, Logistic Regression was selected as the final sentiment model.

## User Based Recommendation System

#### Create User Item Matrix

In [40]:
user_item_matrix = reviews_df.pivot_table(
    index='reviews_username',
    columns='id',
    values='reviews_rating'
)


In [41]:
# Fill in Missing Values
user_item_matrix_filled = user_item_matrix.fillna(0)

In [42]:
user_similarity = cosine_similarity(user_item_matrix_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.index
)

In [43]:
# Find Similar Users

def get_similar_users(user, top_n=10):
    return (
        user_similarity_df[user]
        .sort_values(ascending=False)
        .iloc[1:top_n+1]
    )


In [53]:
product_id_to_name = (
    reviews_df[['id', 'name']]
    .drop_duplicates()
    .set_index('id')['name']
    .to_dict()
)


This function:

- Finds similar users

- Collects products they liked

- Excludes already-rated products

- Returns top 20 recommendations

In [54]:
def recommend_products_with_names(user, top_n=20):
    
    if user not in user_item_matrix.index:
        return "User not found"

    similar_users = get_similar_users(user)
    
    user_rated_products = user_item_matrix.loc[user].dropna().index

    recommendations = {}

    for sim_user, similarity_score in similar_users.items():
        sim_user_ratings = user_item_matrix.loc[sim_user].dropna()

        for product, rating in sim_user_ratings.items():
            if product not in user_rated_products:
                recommendations[product] = recommendations.get(product, 0) + (rating * similarity_score)

    recommended_products = sorted(
        recommendations.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Convert product IDs to product names
    product_names = [
        product_id_to_name.get(prod_id, "Unknown Product")
        for prod_id, score in recommended_products[:top_n]
    ]

    return product_names


In [67]:
sample_user = reviews_df['reviews_username'].iloc[0]

top_20_product_names_ub = recommend_products_with_names(sample_user, top_n=20)
top_20_product_names_ub


[]

### Item Based Recommendation System

In [56]:
# Item User matrix

item_user_matrix = reviews_df.pivot_table(
    index='id',
    columns='reviews_username',
    values='reviews_rating'
)


In [57]:
item_user_matrix_filled = item_user_matrix.fillna(0)

In [58]:
item_user_matrix

reviews_username,00dog3,00sab00,01impala,02dakota,02deuce,0325home,06stidriver,08dallas,09mommy11,1.11E+24,...,zt313,zubb,zulaa118,zuttle,zwithanx,zxcsdfd,zxjki,zyiah4,zzdiane,zzz1127
id,,,,,,,,,,,,,,,,,,,,,
AV13O1A8GV-KLJ3akUyj,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AV14LG0R-jtxr-f38QfS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AV16khLE-jtxr-f38VFn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AV1YGDqsGV-KLJ3adc-O,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AV1YIch7GV-KLJ3addeG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AVpfshNsLJeJML43CB8q,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AVpfthSailAPnD_xg3ON,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AVpftikC1cnluZ0-p31V,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
item_similarity = cosine_similarity(item_user_matrix_filled)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix_filled.index,
    columns=item_user_matrix_filled.index
)

In [60]:
def get_similar_items(item_id, top_n=10):
    return (
        item_similarity_df[item_id]
        .sort_values(ascending=False)
        .iloc[1:top_n+1]
    )


In [61]:
def item_based_recommendation(user, top_n=20):
    
    if user not in item_user_matrix.columns:
        return "User not found"

    user_ratings = item_user_matrix[user].dropna()
    
    recommendations = {}

    for item_id, rating in user_ratings.items():
        if rating >= 4:  # focus on liked items
            similar_items = item_similarity_df[item_id]

            for sim_item, similarity_score in similar_items.items():
                if sim_item not in user_ratings.index:
                    recommendations[sim_item] = recommendations.get(sim_item, 0) + (similarity_score * rating)

    recommended_items = sorted(
        recommendations.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [item for item, score in recommended_items[:top_n]]


In [62]:
def item_based_recommendation_with_names(user, top_n=20):
    item_ids = item_based_recommendation(user, top_n)
    
    return [
        product_id_to_name.get(item_id, "Unknown Product")
        for item_id in item_ids
    ]


In [66]:
sample_user = reviews_df['reviews_username'].iloc[0]

top_20_items_item_based = item_based_recommendation_with_names(sample_user)
top_20_items_item_based


["Nearly Natural 5.5' Bamboo W/decorative Planter",
 'Spam Single Classic',
 'Home Health Hairever Shampoo',
 'Diet Canada Dry Ginger Ale - 12pk/12 Fl Oz Cans',
 'Tostitos Bite Size Tortilla Chips',
 'Chips Ahoy! Original Chocolate Chip - Cookies - Family Size 18.2oz',
 "Meguiar's Ultimate Quik Detailer 22-Oz.",
 "Various - Country's Greatest Gospel:Gold Ed (cd)",
 'Godzilla 3d Includes Digital Copy Ultraviolet 3d/2d Blu-Ray/dvd',
 'Jolly Time Select Premium Yellow Pop Corn',
 'Mike Dave Need Wedding Dates (dvd + Digital)',
 'Planes: Fire Rescue (2 Discs) (includes Digital Copy) (blu-Ray/dvd)',
 'The Resident Evil Collection 5 Discs (blu-Ray)',
 'Chobani174 Strawberry On The Bottom Non-Fat Greek Yogurt - 5.3oz',
 'My Big Fat Greek Wedding 2 (blu-Ray + Dvd + Digital)',
 'Pleasant Hearth 7.5 Steel Grate, 30 5 Bar - Black',
 'Nexxus Exxtra Gel Style Creation Sculptor',
 "Stargate (ws) (ultimate Edition) (director's Cut) (dvdvideo)",
 'Hormel Chili, No Beans',
 "Jason Aldean - They Don't K

## Conclusion - As we can clearly see that for the first user who loved Music, Hip hop music, Rap music, User based recommendation system did not recommend any products for him, however using Item Based Recommendation System recommended the products similar to the user's taste

In [68]:
'joshua' in reviews_df['reviews_username'].unique()


True

In [69]:
user_name = 'joshua'

top_20_recommendations = item_based_recommendation_with_names(
    user=user_name,
    top_n=20
)

top_20_recommendations

["Nearly Natural 5.5' Bamboo W/decorative Planter",
 'Spam Single Classic',
 'Home Health Hairever Shampoo',
 'Diet Canada Dry Ginger Ale - 12pk/12 Fl Oz Cans',
 'Tostitos Bite Size Tortilla Chips',
 'Chips Ahoy! Original Chocolate Chip - Cookies - Family Size 18.2oz',
 "Meguiar's Ultimate Quik Detailer 22-Oz.",
 "Various - Country's Greatest Gospel:Gold Ed (cd)",
 'Godzilla 3d Includes Digital Copy Ultraviolet 3d/2d Blu-Ray/dvd',
 'Jolly Time Select Premium Yellow Pop Corn',
 'Mike Dave Need Wedding Dates (dvd + Digital)',
 'Planes: Fire Rescue (2 Discs) (includes Digital Copy) (blu-Ray/dvd)',
 'The Resident Evil Collection 5 Discs (blu-Ray)',
 'Chobani174 Strawberry On The Bottom Non-Fat Greek Yogurt - 5.3oz',
 'My Big Fat Greek Wedding 2 (blu-Ray + Dvd + Digital)',
 'Pleasant Hearth 7.5 Steel Grate, 30 5 Bar - Black',
 'Nexxus Exxtra Gel Style Creation Sculptor',
 "Stargate (ws) (ultimate Edition) (director's Cut) (dvdvideo)",
 'Hormel Chili, No Beans',
 "Jason Aldean - They Don't K

In [70]:
recommended_df = reviews_df[reviews_df['name'].isin(top_20_recommendations)]


In [73]:
recommended_df['combined_review'] = (
    recommended_df['reviews_title'] + " " + recommended_df['reviews_text']
)

recommended_df['clean_review'] = recommended_df['combined_review'].apply(clean_text)


/var/folders/cr/ksm67q7x18ngxj1221cqjwsw0000gn/T/ipykernel_29918/823702777.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recommended_df['combined_review'] = (
/var/folders/cr/ksm67q7x18ngxj1221cqjwsw0000gn/T/ipykernel_29918/823702777.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recommended_df['clean_review'] = recommended_df['combined_review'].apply(clean_text)


In [74]:
X_recommended_tfidf = tfidf.transform(recommended_df['clean_review'])

recommended_df['positive_sentiment_prob'] = lr_model.predict_proba(
    X_recommended_tfidf
)[:, 1]


/var/folders/cr/ksm67q7x18ngxj1221cqjwsw0000gn/T/ipykernel_29918/3873233033.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recommended_df['positive_sentiment_prob'] = lr_model.predict_proba(


In [75]:
product_sentiment_scores = (
    recommended_df
    .groupby('name')['positive_sentiment_prob']
    .mean()
    .sort_values(ascending=False)
)


In [76]:
top_5_products = product_sentiment_scores.head(5)
top_5_products


name
My Big Fat Greek Wedding 2 (blu-Ray + Dvd + Digital)                   0.839026
Stargate (ws) (ultimate Edition) (director's Cut) (dvdvideo)           0.820517
Jolly Time Select Premium Yellow Pop Corn                              0.776577
Planes: Fire Rescue (2 Discs) (includes Digital Copy) (blu-Ray/dvd)    0.774777
Godzilla 3d Includes Digital Copy Ultraviolet 3d/2d Blu-Ray/dvd        0.758943
Name: positive_sentiment_prob, dtype: float64

## Conclusion - "An item-based collaborative filtering model was used to recommend 20 products for the selected user. These recommendations were further refined using a sentiment analysis model (Logistic Regression with TF-IDF). The average positive sentiment probability of reviews for each recommended product was computed, and the top 5 products with the highest sentiment scores were finally recommended."